In [ ]:
from pathlib import Path

import logfire
from treant.treant import process_document

from medici.common.cache.embedding_cache import EmbeddingCache
from medici.common.services.qdrant import QdrantStorageService
from medici.common.storage.storage_factory import StorageFactory
from medici.common.utils.config import config
from medici.common.utils.constants import ParseMethod, StorageType
from medici.common.utils.helper import separate_content
from medici.common.utils.tokenizer import TikTokenTokenizer
from medici.ingestion.embedding import EmbeddingService
from medici.ingestion.processor import Processor

In [ ]:
logfire.configure(service_name="parsing")

In [ ]:
cwd = Path.cwd().parent
file_path = cwd / "data/Attention-is_all_you_need.pdf"

In [ ]:
local_config = {
    "type": StorageType.LOCAL.value,
    "base_dir": config.STORAGE_BASE_DIR,
}

in_storage = StorageFactory.create(local_config)

In [ ]:
tokenizer = TikTokenTokenizer()
emb_cache = await EmbeddingCache.create(dsn=config.POSTGRES_CONN_STRING, max_entries=50_000)
embedding_service = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME,
    dimensions=config.EMBEDDING_DIMENSIONS,
    batch_size=config.EMBEDDING_BATCH_SIZE,
    cache=emb_cache,
)

storage_service = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=embedding_service.vector_size,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [ ]:
processor = Processor(tokenizer, embedding_service, storage_service)

In [ ]:
out, doc_id = await process_document(file_path=str(file_path), parse_method=ParseMethod.DOCLING)

In [ ]:
print(out)

In [ ]:
content_list, multimodal_items, text_blocks = separate_content(out)

In [ ]:
multimodal_items

In [ ]:
chunk_context = await processor._chunk_doc_content(
    file_path=file_path,
    content_list=content_list,
    multimodal_items=multimodal_items,
    doc_id=doc_id,
    parse_method=ParseMethod.DOCLING,
    text_blocks=text_blocks,
)

In [ ]:
result = await processor.ingest_document(file_path=file_path, parse_method=ParseMethod.DOCLING)

In [ ]:
result